# Building Generative Pretrained Transformer (GPT) From Scratch

In [1]:
# Importing Dependencies
import tiktoken
import warnings
import torch
import yaml
import sys

In [2]:
# Filtering out potential warnings
warnings.filterwarnings(action="ignore")

In [3]:
# Loading the configuration
with open(file="../Configuration/config.yml") as yaml_file:
    config=yaml.safe_load(stream=yaml_file)
    
# Extracting the configuration for GPT (Original)
llm_config=config["gpt"]["small"]

# Displaying the configurations for GPT (Original)
llm_config

{'vocab_size': 50257,
 'context_length': 1024,
 'emb_dim': 768,
 'n_heads': 12,
 'n_layers': 12,
 'drop_rate': 0.1,
 'qkv_bias': True}

In [4]:
# Defining a device name
device_name="cuda" if torch.cuda.is_available() else "cpu"

# Specifying the device
device=torch.device(device=device_name)

# Making sure the GPU is being utilized
assert torch.cuda.is_available()

### Layer Normalization

In [5]:
# Designing a custom Layer Normalization (LayerNorm)
class LayerNorm(torch.nn.Module):
    # Defining instance attribute(s)
    def __init__(self, emb_dim: int, eps: float=1e-5):
        # Inheriting the methods and attributes
        super().__init__()
        
        # Defining the shift parameter
        self.shift=torch.nn.Parameter(data=torch.zeros(emb_dim))
        
        # Defining the scale parameter
        self.scale=torch.nn.Parameter(data=torch.ones(emb_dim))
        
        # Defining the epsilon parameter
        self.eps=eps
    
    # Defining the forward instance method
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Calculating the average of the tensor
        mean=x.mean(dim=-1, keepdim=True)
        
        # Calculating the variance of the tensor
        var=x.var(dim=-1, unbiased=False, keepdim=True)
        
        # Normalizing the values
        norm_x=(x-mean)/torch.sqrt(input=var+self.eps)
        
        # Adding the scale and shift parameters
        norm_x=(norm_x*self.scale)+self.shift
        
        # Returning the normalized values
        return norm_x

In [6]:
# Creating a sample data
sample_data=torch.randn(size=[2, 16, llm_config["emb_dim"]])

# Applying normalization
LayerNorm(emb_dim=llm_config["emb_dim"])(sample_data)

tensor([[[-1.9592, -0.3348,  0.7269,  ..., -1.0878, -0.8707,  0.1603],
         [ 1.9423, -0.2259, -1.3590,  ..., -0.0983,  0.5365,  0.3493],
         [-1.3775,  1.6772,  0.2328,  ..., -0.7277, -1.0446,  0.1072],
         ...,
         [ 1.8622,  0.1154,  0.2481,  ...,  1.2326,  0.9456,  0.4730],
         [ 1.5305, -0.7402,  0.1872,  ..., -0.6130, -0.6181,  0.5328],
         [ 0.9306,  0.3590,  0.8633,  ..., -0.5851, -2.1484, -0.8476]],

        [[ 2.0820,  0.4665,  0.4693,  ..., -0.0224, -0.6324,  1.0368],
         [-0.0220, -2.0739,  0.1216,  ...,  0.2421, -0.1501,  0.0277],
         [ 1.2544,  1.3111, -1.5394,  ..., -0.0533,  1.5897,  0.7737],
         ...,
         [ 1.0270, -3.2459, -0.0754,  ...,  2.0040,  0.2365,  0.3991],
         [-0.7228, -0.5381, -0.0246,  ..., -1.5928, -0.7625, -0.3056],
         [ 0.4902, -1.0825,  0.0548,  ..., -1.1937,  0.8426,  0.3953]]],
       grad_fn=<AddBackward0>)

In [7]:
# Checking if both tensors are same
torch.allclose(LayerNorm(emb_dim=llm_config["emb_dim"])(sample_data),
               torch.nn.LayerNorm(normalized_shape=llm_config["emb_dim"])(sample_data))

True

### Gaussian Error Linear Unit (GELU) Activation Function

In [8]:
# Designing a custom GELU activation function
class GELU(torch.nn.Module):
    # Defining instance attributes
    def __init__(self):
        # Inheriting methods and attributes
        super().__init__()
        
    # Defining the forward instance method
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Calculating the gelu
        gelu=0.5*x*(1+torch.tanh(input=torch.sqrt(torch.tensor(data=2/torch.pi)) * (x+(0.044715*torch.pow(input=x, exponent=3)))))
        
        # Returning the gelu
        return gelu

In [9]:
# Applying the activation function
GELU()(sample_data)

tensor([[[-4.2066e-02, -1.3029e-01,  5.6137e-01,  ..., -1.4505e-01,
          -1.6480e-01,  8.1970e-02],
         [ 1.9045e+00, -8.1308e-02, -1.2441e-01,  ..., -3.1041e-02,
           4.0126e-01,  2.4425e-01],
         [-1.2138e-01,  1.6450e+00,  1.6646e-01,  ..., -1.6908e-01,
          -1.5848e-01,  8.2800e-02],
         ...,
         [ 1.8030e+00,  5.2688e-02,  1.3709e-01,  ...,  1.0905e+00,
           7.7238e-01,  3.1049e-01],
         [ 1.4268e+00, -1.6959e-01,  8.3798e-02,  ..., -1.6847e-01,
          -1.6866e-01,  3.4869e-01],
         [ 8.1022e-01,  2.3345e-01,  7.3404e-01,  ..., -1.6714e-01,
          -2.4217e-02, -1.6467e-01]],

        [[ 2.1370e+00,  3.7999e-01,  3.8251e-01,  ...,  2.3616e-02,
          -1.6210e-01,  9.6660e-01],
         [-1.1850e-02, -4.1567e-02,  6.4111e-02,  ...,  1.4003e-01,
          -6.6198e-02,  1.2635e-02],
         [ 1.1465e+00,  1.2096e+00, -1.0164e-01,  ..., -9.4400e-03,
           1.5206e+00,  6.3010e-01],
         ...,
         [ 8.8065e-01, -1

In [10]:
# Checking if both tensors are same
torch.allclose(input=GELU()(sample_data),
               other=torch.nn.GELU(approximate="tanh")(sample_data))

True

### Multi Head Attention (MHA) Mechanism

#### Inputs:

- n_heads: The number of attention heads
- qkv_bias: The flag whether to apply bias for vector calculation
- d_model: The dimension (input & output) used for vector calculation
- context_length: The maximum number of tokens an LLM can process
- dropout_proba: The probability threshold used for regularization

#### Outputs:

- Context vectors with regularized causal masked attention weights

In [11]:
# Designing a multi head attention (MHA) mechanism
class MultiHeadAttention(torch.nn.Module):
    # Defining the instance attribute(s)
    def __init__(self,
                 d_model: int,
                 n_heads: int,
                 qkv_bias: bool,
                 context_length: int,
                 dropout_proba: float):
        # Inheriting the methods and attributes
        super().__init__()
        
        # Asserting the dimension to be divisible by the number of attention heads
        assert d_model % n_heads == 0
        
        # Defining the causal masked attention mechanism
        self.register_buffer(name="mask", tensor=torch.triu(input=torch.ones(size=[context_length, context_length], dtype=torch.bool), diagonal=1))
        
        # Defining the linear layer for the calculation of query, key, and value (qkv) vectors
        self.W_qkv=torch.nn.Linear(in_features=d_model, out_features=d_model*3, bias=qkv_bias)
        
        # Defining the linear projections layer that are necessary for LLMs
        self.linear_proj=torch.nn.Linear(in_features=d_model, out_features=d_model)
        
        # Defining the regularization
        self.dropout=torch.nn.Dropout(p=dropout_proba)
        
        # Defining the number of dimensions for the attention heads
        self.head_dim=d_model // n_heads
        
        # Defining the dimension (input & output)
        self.d_model=d_model
        
        # Defining the number of attention heads
        self.n_heads=n_heads
        
    # Defining the forward instance method
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Extracting the number of batches, and tokens
        n_batches, n_tokens, _ = x.shape
        
        # Instantiating the query, key, and value vectors
        qkv=self.W_qkv(x)
        
        # Unpacking the tensor to extract the query, key and value vectors
        queries, keys, values=torch.chunk(input=qkv, chunks=3, dim=-1)
        
        # Reshaping the query vectors from [b, t, d] to [b, t, h, d]
        queries=queries.view(n_batches, n_tokens, self.n_heads, self.head_dim)
        
        # Reshaping the value vectors from [b, t, d] to [b, t, h, d]
        values=values.view(n_batches, n_tokens, self.n_heads, self.head_dim)
        
        # Reshaping the key vectors from [b, t, d] to [b, t, h, d]
        keys=keys.view(n_batches, n_tokens, self.n_heads, self.head_dim)
        
        # Tranposing the query vectors from [b, t, h, d] to [b, h, t, d]
        queries=queries.transpose(1, 2)
        
        # Tranposing the value vectors from [b, t, h, d] to [b, h, t, d]
        values=values.transpose(1, 2)
        
        # Tranposing the key vectors from [b, t, h, d] to [b, h, t, d]
        keys=keys.transpose(1, 2)
        
        # Calculating the attentions scores
        attn_scores=queries @ keys.transpose(-2, -1)
        
        # Instantiating the masking for the causal attention mechanism
        masking=self.mask[:n_tokens, :n_tokens]
        
        # Applying the masking to attention scores
        attn_scores_masked=attn_scores.masked_fill(mask=masking, value=-torch.inf)
        
        # Calculating the regularized attention weights
        attn_weights=self.dropout(torch.softmax(input=attn_scores_masked/self.head_dim**0.5, dim=-1))
        
        # Calculating the context vectors
        context_vectors=(attn_weights @ values).transpose(1, 2)
        
        # Reshaping the context vectors and applying linear projections
        context_vectors=self.linear_proj(context_vectors.contiguous().view(n_batches, n_tokens, self.d_model))
        
        # Returning the context vectors
        return context_vectors

In [12]:
# Creating a dictionary of keyword arguments
llm_kwargs=dict(d_model=llm_config["emb_dim"],
                n_heads=llm_config["n_heads"],
                qkv_bias=llm_config["qkv_bias"],
                context_length=llm_config["context_length"],
                dropout_proba=llm_config["drop_rate"])

# Instantiating the MHA
mha=MultiHeadAttention(**llm_kwargs)

# Sending the sample data to MHA
output=mha(sample_data)

# Displaying the shape and output
display(output.shape, output)

torch.Size([2, 16, 768])

tensor([[[ 0.1911,  0.1859,  0.0245,  ..., -0.2114, -0.4089, -0.2974],
         [ 0.1913,  0.2302, -0.5023,  ..., -0.0742, -0.1672, -0.4572],
         [ 0.2437,  0.0136, -0.4366,  ...,  0.1005, -0.1328, -0.4037],
         ...,
         [ 0.0733,  0.2895, -0.1805,  ..., -0.2328, -0.0500, -0.2105],
         [ 0.1397,  0.1822, -0.1237,  ..., -0.1243, -0.0009, -0.1572],
         [ 0.1047,  0.1477, -0.2067,  ..., -0.1598,  0.0560, -0.2018]],

        [[ 0.0632, -0.3183,  0.2585,  ...,  0.0358, -0.3357, -0.0250],
         [-0.1421,  0.0529, -0.1636,  ..., -0.2927, -0.4531, -0.4719],
         [-0.3775,  0.1049,  0.0222,  ..., -0.3288, -0.4749, -0.3234],
         ...,
         [-0.1165,  0.1352,  0.0440,  ..., -0.0357, -0.1882, -0.0850],
         [-0.1546,  0.1461,  0.0241,  ..., -0.0719, -0.2357, -0.1291],
         [-0.1449,  0.0790,  0.1147,  ...,  0.0940, -0.1529, -0.1146]]],
       grad_fn=<ViewBackward0>)

### Feed Forward Neural Network

In [13]:
# Designing the Feed Forward Neural Network
class FeedForward(torch.nn.Module):
    # Defining the instance attribute(s)
    def __init__(self, emb_dim: int):
        # Inheriting the methods and attributes
        super().__init__()
        
        # Defining the input layer
        input_layer=torch.nn.Linear(in_features=emb_dim, out_features=emb_dim*4)
        
        # Defining the output layer
        output_layer=torch.nn.Linear(in_features=emb_dim*4, out_features=emb_dim)
        
        # Defining the activation function
        gelu=GELU()
        
        # Defining a sequential neural network
        self.layers=torch.nn.Sequential(input_layer, gelu, output_layer)
        
    # Defining the forward instance method
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Calculating the hidden states
        hidden_states=self.layers(x)
        
        # Returning the hidden states
        return hidden_states

### Transformer (Decoder Only)

In [14]:
# Designing a transformer architecture
class TransformerBlock(torch.nn.Module):
    # Defining the instance attributes
    def __init__(self, config: dict):
        # Inheriting the methods and attributes
        super().__init__()
        
        # Creating a dictionary of keyword arguments
        mha_kwargs=dict(d_model=config["emb_dim"],
                        n_heads=config["n_heads"],
                        qkv_bias=config["qkv_bias"],
                        context_length=config["context_length"],
                        dropout_proba=config["drop_rate"])
        
        # Instantiating the multi head attention
        self.mha=MultiHeadAttention(**mha_kwargs)
        
        # Instantiating the feed forward neural network
        self.ff=FeedForward(emb_dim=config["emb_dim"])
        
        # Instantiating the layer normalization
        self.layer_norm_mha=LayerNorm(emb_dim=config["emb_dim"])
        
        # Instantiating the layer normalization
        self.layer_norm_ff=LayerNorm(emb_dim=config["emb_dim"])
        
        # Instantiating the regularization
        self.dropout=torch.nn.Dropout(p=config["drop_rate"])
        
    # Defining the forward instance method
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Initializing the shortcut tensor
        shortcut=x
        
        # Applying normalization
        x=self.layer_norm_mha(x)
        
        # Calculating the context vectors
        x=self.mha(x)
        
        # Applying regularization
        x=self.dropout(x)
        
        # Updating the tensors with shortcut values
        x=x+shortcut
        
        # Reinitializing the shortcut with updated values
        shortcut=x
        
        # Applying normalization
        x=self.layer_norm_ff(x)
        
        # Calculating the hidden states
        x=self.ff(x)
        
        # Applying regularization
        x=self.dropout(x)
        
        # Updating the tensors with shortcut values
        x=x+shortcut
        
        # Returning the values
        return x

### GPT Model

In [15]:
# Designing the GPT model
class GPTModel(torch.nn.Module):
    # Defining instance attributes
    def __init__(self, config: dict):
        # Inheriting the methods and attributes
        super().__init__()
        
        # Defining the token embedding layer
        self.tok_emb_layer=torch.nn.Embedding(num_embeddings=config["vocab_size"], embedding_dim=config["emb_dim"])
        
        # Defining the positional (absolute) embedding layer
        self.pos_emb_layer=torch.nn.Embedding(num_embeddings=config["context_length"], embedding_dim=config["emb_dim"])
        
        # Defining the transformer blocks
        self.trf_blocks=torch.nn.Sequential(*[TransformerBlock(config=config) for _ in range(config["n_layers"])])
        
        # Defining the linear projections that are necessary for LLMs
        self.lm_head=torch.nn.Linear(in_features=config["emb_dim"], out_features=config["vocab_size"], bias=False)
        
        # Defining the layer normalization
        self.final_layer_norm=LayerNorm(emb_dim=config["emb_dim"])
        
        # Defining the regularization
        self.dropout=torch.nn.Dropout(p=config["drop_rate"])
        
    # Defining the forward method
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Unpacking the values of the tensor
        _, seq_len=x.shape
        
        # Calculating the token embeddings based on the inputs
        tok_embs=self.tok_emb_layer(x)
        
        # Calculating the positional (absolute) embeddings based on the sequence length
        pos_embs=self.pos_emb_layer(torch.arange(seq_len, device=x.device))
        
        # Calculating the combined embeddings
        x=tok_embs+pos_embs
        
        # Applying regularization
        x=self.dropout(x)
        
        # Calculating hidden states
        x=self.trf_blocks(x)
        
        # Applying the final normalization
        x=self.final_layer_norm(x)
        
        # Mapping hidden states to vocabulary logits
        x=self.lm_head(x)
        
        # Returning the logits
        return x

In [16]:
# Loading the tokenizer
tokenizer=tiktoken.get_encoding(encoding_name="gpt2")

# Creating a sample sentence
sample="I like watching movies"

# Creating a batch
sample_tensor=torch.tensor(data=tokenizer.encode(text=sample, allowed_special={"<|endoftext|>"})).unsqueeze(dim=0)

# Instantiating the GPT model
model=GPTModel(config=llm_config)

# Calculating the logits
logits=model(sample_tensor)

# Displaying the logits
logits

tensor([[[-0.2070,  0.2857,  1.5039,  ...,  2.0273, -0.2917, -0.4711],
         [ 0.5177,  0.8975, -0.0293,  ..., -0.0312, -0.3060, -0.6432],
         [ 0.0425, -0.0750,  0.1420,  ..., -0.5668,  1.1690,  0.8274],
         [ 0.3487, -0.2529,  1.2575,  ...,  0.1421, -0.2892, -0.0187]]],
       grad_fn=<UnsafeViewBackward0>)

#### Generating text

In [17]:
# Appending the path
sys.path.append("../Scripts/")

# Importing the function to generate text
from utils import generate_text

# Fixing the seed for reproducibility
torch.manual_seed(123)

# Defining the end of sequence ID (EOS)
eos_id=tokenizer.encode(text="<|endoftext|>", allowed_special={"<|endoftext|>"})[0]

# Defining a sample text
sample_text="He never wanted to"

# Converting the text into token IDs
sample_token_ids=torch.tensor(tokenizer.encode(text=sample_text, allowed_special={"<|endoftext|>"})).unsqueeze(dim=0)

# Generating the next tokens
generated_token_ids=generate_text(model=model,
                                  token_ids=sample_token_ids,
                                  max_new_tokens=5,
                                  context_length=llm_config["context_length"],
                                  temperature=0.5,
                                  top_k=50,
                                  eos_id=eos_id)

# Displaying the generated token IDs
generated_token_ids

tensor([[ 1544,  1239,  2227,   284, 35716, 20243, 25090, 31900, 33452]])

In [18]:
# Printing out the generated token IDs
print(tokenizer.decode(tokens=generated_token_ids.squeeze().tolist()))

He never wanted to Umb workshop 162 Angry Canberra


### Loading Pretrained GPT-2 Weights

In [19]:
# Loading the functions
from utils import download_and_load_gpt2, load_weights_into_gpt

# Downloading the GPT-2 (Small) weights
settings, params=download_and_load_gpt2(model_size="124M")

# Extracting the matching configuration for GPT-2 (Small)
gpt2_config=config["gpt"]["small"]

# Deleting the GPT model
del GPTModel

# Loading the GPT model from the script
from architecture import GPTModel

# Instantiating the GPT model
gpt2=GPTModel(config=gpt2_config)

# Loading the pretrained weights
load_weights_into_gpt(gpt=gpt2, params=params)

# Moving the model to the device and switching to evaluation mode
gpt2=gpt2.to(device=device).eval()

# Displaying the settings
settings

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

{'n_vocab': 50257, 'n_ctx': 1024, 'n_embd': 768, 'n_head': 12, 'n_layer': 12}

In [20]:
# Fixing the seed for reproducibility
torch.manual_seed(123)

# Converting the text into token IDs
pretrained_token_ids=torch.tensor(tokenizer.encode(text="He never wanted to")).unsqueeze(dim=0).to(device=device)

# Generating the next tokens with the pretrained model
pretrained_token_ids=generate_text(model=gpt2,
                                   token_ids=pretrained_token_ids,
                                   max_new_tokens=25,
                                   context_length=gpt2_config["context_length"],
                                   temperature=1.5,
                                   top_k=50,
                                   eos_id=eos_id)

# Printing out the generated text
print(tokenizer.decode(tokens=pretrained_token_ids.squeeze().tolist()))

He never wanted to see his wife or kids. And I was told that she wouldn't give up their right to a vote; even if I
